# Real-Time Multimodal Emotion Prototype — MELD (Text + Vision, extended to Text + Vision + Audio)

Character-robot prototype: each conversational **utterance** (text + face + voice) is turned into a
**structured MELD emotion tag** plus a **short grounded response**, within a per-utterance latency budget.

**Run this on Kaggle with a GPU accelerator (T4 x2 recommended, T4 x1 is enough).**
Settings (right panel) -> Accelerator -> GPU T4 x2.

### What this notebook does
1. Installs deps and loads this repo's `src/` code (attach it as a Kaggle Dataset — see next cell).
2. Loads a subsample of MELD, extracts keyframes + audio per utterance with `ffmpeg`.
3. Encodes each utterance with frozen pretrained encoders (text / vision / audio).
4. Trains the small trainable **fusion head** (structured-tag classifier) — supervised, cross-entropy.
5. Trains the **RL-gated fusion** (REINFORCE policy over modality trust) on top of the frozen fusion head —
   this is the RL component applied to one part of the system.
6. Reports **static-fusion vs RL-gated-fusion accuracy/F1** — the measured value of the RL component.
7. Reports a **Text+Vision-only vs full tri-modal** ablation — evidence for the core track and the extension.
8. Runs the **streaming demo**: one MELD dialogue, utterance-by-utterance, each producing a structured tag
   + grounded response text, with per-utterance latency and peak GPU memory reported.
9. Saves the trained (tiny) fusion/gate checkpoints to `/kaggle/working/artifacts/` for local reuse.

See the repo `README.md` for the real-time definition, architecture rationale, parameter-budget accounting,
and what was intentionally left out.


## 0. Attach this repository's code and the MELD dataset

This notebook imports `src/` from this repository rather than duplicating ~500 lines of code inline, to
avoid the notebook and the repo drifting out of sync.

1. Push/upload this whole repository folder as a **Kaggle Dataset** (Kaggle -> Create -> New Dataset ->
   upload the folder), or as a **Utility Script**. Note its slug.
2. In this notebook: **Add Data** -> attach that dataset, and also attach
   [zaber666/meld-dataset](https://www.kaggle.com/datasets/zaber666/meld-dataset) (the MELD mirror this
   notebook is configured for).
3. `MELD_ROOT` below assumes that mirror's layout: `<root>/train/train_sent_emo.csv` +
   `<root>/train/train_splits/*.mp4` (and similarly for `dev`/`test`) — i.e. each split's CSV and videos
   live inside a same-named subfolder, per `src/data_meld.py`. If Kaggle mounts it under a different
   folder name than `meld-dataset`, or the layout doesn't match once you can see it in the notebook's
   file browser, fix `MELD_ROOT` and/or `src/data_meld.py`'s `SPLIT_CSVS`/`SPLIT_VIDEO_DIRS` accordingly
   — this was set from a collaborator's inspection of the dataset, not independently verified here.


In [ ]:
import sys, os

REPO_DIR = "/kaggle/input/<your-repo-dataset-slug>"   # <-- set this to your uploaded repo dataset
MELD_ROOT = "/kaggle/input/meld-dataset"               # zaber666/meld-dataset, mounted by its slug

assert os.path.exists(REPO_DIR), "attach the repo dataset and fix REPO_DIR above"
assert os.path.exists(MELD_ROOT), "attach zaber666/meld-dataset (or fix MELD_ROOT to match how it mounted)"
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("repo:", REPO_DIR)
print("meld:", MELD_ROOT)


In [ ]:
!pip install -q -r requirements.txt


In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device count:", torch.cuda.device_count())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()


## 1. Pick a model-size config and check the parameter budget (constraint: <= 6B total)

In [ ]:
MODEL_SIZE = "small"   # "small" (~1.7B) or "large" (~3.3B) -- both are well under the 6B cap
CONFIG_PATH = f"configs/{MODEL_SIZE}.yaml"

!python -m src.param_budget approx --config {CONFIG_PATH}


## 2. Load a MELD subsample and extract per-utterance keyframes + audio

Subsampled for a fast, correct prototype within the 2-3 day timebox (see README trade-offs). Raise
`N_TRAIN`/`N_TEST` if you have GPU-hours to spare — the pipeline itself is not limited to this subsample.

In [ ]:
from src.data_meld import load_split, extract_keyframes, extract_audio

N_TRAIN = 600   # utterances
N_TEST = 200

train_utts = load_split(MELD_ROOT, "train", limit=N_TRAIN)
test_utts = load_split(MELD_ROOT, "test", limit=N_TEST)
print(len(train_utts), "train utterances,", len(test_utts), "test utterances")
train_utts[0]


In [ ]:
from src.encoders import TextEncoder, VisionEncoder, AudioEncoder
from src.fusion import concat_embeddings, MODALITY_ORDER
from src.schema import MELD_EMOTIONS
import yaml, torch as T

cfg = yaml.safe_load(open(CONFIG_PATH))
embed_dims = cfg["fusion"]["embed_dims"]

text_enc = TextEncoder(cfg["encoders"]["text"]["checkpoint"], device=DEVICE)
vision_enc = VisionEncoder(cfg["encoders"]["vision"]["checkpoint"], device=DEVICE)
audio_enc = AudioEncoder(cfg["encoders"]["audio"]["checkpoint"], device=DEVICE)

def embed_utterances(utts, n_frames=3):
    X, y, skipped = [], [], 0
    for u in utts:
        try:
            frames = extract_keyframes(u.video_path, n_frames=n_frames)
            wav = extract_audio(u.video_path)
            embeds = {
                "text": text_enc.encode(u.text),
                "vision": vision_enc.encode(frames),
                "audio": audio_enc.encode(wav),
            }
        except Exception as e:
            skipped += 1
            continue
        X.append(concat_embeddings(embeds, embed_dims))
        y.append(MELD_EMOTIONS.index(u.emotion))
    print(f"embedded {len(X)} utterances, skipped {skipped} (missing/corrupt clips)")
    return T.stack(X), T.tensor(y, dtype=T.long)

X_train, y_train = embed_utterances(train_utts)
X_test, y_test = embed_utterances(test_utts)
X_train.shape, X_test.shape


## 3. Train the fusion head (structured-tag classifier) — the static/baseline fusion

In [ ]:
from src.fusion import FusionHead, train_fusion_head

fcfg = cfg["fusion"]
fusion = FusionHead(embed_dims, fcfg["hidden_dim"], fcfg["num_classes"])
loss_hist = train_fusion_head(fusion, X_train, y_train, epochs=60, lr=1e-3)

import matplotlib.pyplot as plt
plt.plot(loss_hist); plt.title("fusion head training loss"); plt.xlabel("epoch"); plt.ylabel("CE loss"); plt.show()


## 4. Train the RL-gated fusion (REINFORCE) — the RL component

The gate policy learns a per-utterance, per-modality trust weight on top of the **frozen** fusion head above,
rewarded for producing a correct classification. Its value is measured next: static vs RL-gated accuracy/F1
on the held-out test split.

In [ ]:
from src.fusion import RLGatePolicy, train_rl_gate, evaluate

gate = RLGatePolicy(embed_dims, fcfg["gate_hidden_dim"])
reward_hist = train_rl_gate(gate, fusion, embed_dims, X_train, y_train, epochs=150, lr=5e-3)

plt.plot(reward_hist); plt.title("RL gate mean reward per batch"); plt.xlabel("epoch"); plt.ylabel("mean reward"); plt.show()


## 5. Measured RL value: static fusion vs RL-gated fusion on held-out test data

In [ ]:
acc_static, f1_static = evaluate(fusion, X_test, y_test, gate=None)
acc_rl, f1_rl = evaluate(fusion, X_test, y_test, gate=gate, embed_dims=embed_dims)

print(f"{'':20s} {'accuracy':>10s} {'macro-F1':>10s}")
print(f"{'static fusion':20s} {acc_static:>10.3f} {f1_static:>10.3f}")
print(f"{'RL-gated fusion':20s} {acc_rl:>10.3f} {f1_rl:>10.3f}")
print(f"delta: {acc_rl - acc_static:+.3f} accuracy, {f1_rl - f1_static:+.3f} macro-F1")


## 6. Ablation: core Text+Vision-only vs full Text+Vision+Audio

Same trained fusion head, audio slice masked to zero at eval time -> this is exactly what the core-track
(Text+Vision) deliverable looks like standalone, from the same checkpoint used for the tri-modal extension.

In [ ]:
X_test_novoice = X_test.clone()
audio_start = embed_dims["text"] + embed_dims["vision"]
X_test_novoice[:, audio_start:] = 0.0

acc_tv, f1_tv = evaluate(fusion, X_test_novoice, y_test, gate=gate, embed_dims=embed_dims)
acc_tva, f1_tva = evaluate(fusion, X_test, y_test, gate=gate, embed_dims=embed_dims)

print(f"{'':24s} {'accuracy':>10s} {'macro-F1':>10s}")
print(f"{'Text+Vision (core)':24s} {acc_tv:>10.3f} {f1_tv:>10.3f}")
print(f"{'Text+Vision+Audio (ext)':24s} {acc_tva:>10.3f} {f1_tva:>10.3f}")


## 7. Save the trained (tiny) checkpoints

In [ ]:
import os
os.makedirs("/kaggle/working/artifacts", exist_ok=True)
T.save(fusion.state_dict(), f"/kaggle/working/artifacts/fusion_{MODEL_SIZE}.pt")
T.save(gate.state_dict(), f"/kaggle/working/artifacts/gate_{MODEL_SIZE}.pt")
print("saved to /kaggle/working/artifacts/ -- download these and copy into the repo's artifacts/ folder")
print("to run demo.py locally with real trained weights.")


## 8. Streaming demo: one MELD dialogue, turn by turn, to both outputs

Loads the small (or large) response LLM and runs the full per-utterance pipeline over one real MELD
conversation in speaking order — simulating turns "arriving over time". Each turn prints the structured
tag (with the RL-learned modality gate) and the grounded response text, plus latency.

In [ ]:
from src.stream_demo import Pipeline, run_meld_dialogue

pipeline = Pipeline(CONFIG_PATH, device=DEVICE,
                     fusion_ckpt=f"/kaggle/working/artifacts/fusion_{MODEL_SIZE}.pt",
                     gate_ckpt=f"/kaggle/working/artifacts/gate_{MODEL_SIZE}.pt")

DIALOGUE_ID = test_utts[0].dialogue_id
latencies = []
for utt, tag in run_meld_dialogue(pipeline, MELD_ROOT, "test", DIALOGUE_ID, use_audio=True):
    latencies.append(tag.latency_ms)
    print(f"[{utt.speaker}] {utt.text!r}  (gold: {utt.emotion})")
    print(f"  -> tag: {tag.emotion} (conf {tag.emotion_confidence:.2f}), gate={tag.modality_gate.as_dict()}")
    print(f"  -> robot: {tag.response_text}")
    print(f"  -> latency: {tag.latency_ms:.0f} ms\n")

print(f"avg latency: {sum(latencies)/len(latencies):.0f} ms over {len(latencies)} utterances")
if torch.cuda.is_available():
    print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")


## 9. Hardware / resource report (fill in from the run above, copy into README)

- GPU: Kaggle T4 (16GB VRAM); notebook only used a single T4 here (see README for why 2xT4 wasn't required
  at this parameter scale).
- Peak GPU memory observed: see cell 8 output above.
- Average per-utterance latency (encode all modalities + classify + generate response): see cell 8 output.
- Wall-clock for embedding N_TRAIN+N_TEST utterances (frame/audio extraction dominates): see cell 2 timing.
- Exact runtime parameter count (loads real weights, run once GPU is warm):
  `!python -m src.param_budget exact --config {CONFIG_PATH}`


In [ ]:
!python -m src.param_budget exact --config {CONFIG_PATH}
